# Notebook 2: Report Generator (Standalone NPO)
This notebook runs as its own NPO in a Task DAG.
It receives parameters via `ARGUMENTS` (passed by the parent Task which reads them from `SYSTEM$GET_PREDECESSOR_RETURN_VALUE`).

**Expected ARGUMENTS (JSON string):**
```
{"output_table": "DB.SCHEMA.ORDER_SUMMARY", "category_count": 5, "run_date": "2026-09-30", "status": "SUCCESS"}
```

In [ ]:
import sys
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import session as snowpark_session

# Handle multiple active sessions in headless mode
try:
    session = get_active_session()
except Exception:
    session = next(iter(snowpark_session._active_sessions))

# Filter out kernel launcher args
user_args = [a for a in sys.argv[1:] if not a.startswith('-') and not a.startswith('/')]
print(f"User args: {user_args}")

if len(user_args) > 0:
    # Parse pipe-delimited: OUTPUT_TABLE|CATEGORY_COUNT|RUN_DATE|STATUS
    raw = user_args[0]
    print(f"Raw argument from predecessor: {raw}")
    parts = raw.split('|')
    source_table = parts[0]
    category_count = parts[1] if len(parts) > 1 else 'unknown'
    run_date = parts[2] if len(parts) > 2 else 'unknown'
    status = parts[3] if len(parts) > 3 else 'SUCCESS'
else:
    source_table = 'AGENTHIVE_LINEAGE.NPO_DEMO.ORDER_SUMMARY'
    category_count = '5'
    run_date = '2026-09-30'
    status = 'SUCCESS'
    print(f"Interactive mode. Using defaults.")

print(f"\nSource table: {source_table}")
print(f"Run date:     {run_date}")
print(f"Prep status:  {status}")

assert status == 'SUCCESS', f"Predecessor failed with status: {status}"

In [ ]:
# Step 1: Read the summary data from Notebook 1's output
report_df = session.sql(f"SELECT * FROM {source_table} ORDER BY TOTAL_REVENUE DESC")
report_df.show()

# Step 2: Compute top-level KPIs
kpis = session.sql(f"""
SELECT 
  SUM(ORDER_COUNT) AS TOTAL_ORDERS,
  SUM(TOTAL_REVENUE) AS GRAND_TOTAL_REVENUE,
  ROUND(AVG(AVG_ORDER_VALUE), 2) AS OVERALL_AVG_ORDER,
  MAX(REPORT_DATE) AS REPORT_DATE
FROM {source_table}
""").collect()[0]

print(f"\n===== ORDER REPORT =====")
print(f"Report Date:      {kpis['REPORT_DATE']}")
print(f"Total Orders:     {kpis['TOTAL_ORDERS']}")
print(f"Total Revenue:    ${kpis['GRAND_TOTAL_REVENUE']:,.2f}")
print(f"Avg Order Value:  ${kpis['OVERALL_AVG_ORDER']:,.2f}")
print(f"========================")

In [ ]:
# Step 3: Write final report table
TARGET_DB = source_table.split('.')[0]
TARGET_SCHEMA = source_table.split('.')[1]

session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.FINAL_REPORT AS
SELECT 
  CATEGORY,
  ORDER_COUNT,
  TOTAL_REVENUE,
  AVG_ORDER_VALUE,
  UNIQUE_CUSTOMERS,
  ROUND(TOTAL_REVENUE / NULLIF(UNIQUE_CUSTOMERS, 0), 2) AS REVENUE_PER_CUSTOMER,
  REPORT_DATE,
  CURRENT_TIMESTAMP() AS GENERATED_AT
FROM {source_table}
ORDER BY TOTAL_REVENUE DESC
""").collect()

print(f"Final report written to {TARGET_DB}.{TARGET_SCHEMA}.FINAL_REPORT")
print("Report notebook completed successfully.")